In [1]:
import anndata as ad
import pandas as pd
import scanpy as sc
import squidpy as sq
import numpy as np
import spatialdata_io as sd

import bin2cell as b2c
import matplotlib.pyplot as plt
import json
import os

/Users/evangrosso/Library/r-miniconda-arm64/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Import Data

In [2]:
# Paths
base_path = "../data/SLV14/binned_outputs/square_002um"
mtx_path = os.path.join(base_path, "filtered_feature_bc_matrix")
spatial_path = os.path.join(base_path, "spatial")

hires_image_path = "../data/SLV14/binned_outputs/square_002um/spatial/tissue_hires_image.png"

# Load counts
adata = sc.read_10x_mtx(
    mtx_path,
    var_names="gene_symbols",
    make_unique=True
)

# Load positions
positions = pd.read_parquet(
    os.path.join(spatial_path, "tissue_positions.parquet")
)

positions = positions.set_index("barcode")

# Make barcode formatting match
adata.obs_names = adata.obs_names.str.replace("-1$", "", regex=True)
positions.index = positions.index.str.replace("-1$", "", regex=True)

# Keep only barcodes that have spatial positions
common = adata.obs_names.intersection(positions.index)

adata = adata[common].copy()
positions = positions.loc[common].copy()

# Add spatial metadata
adata.obs["in_tissue"] = positions["in_tissue"].to_numpy()
adata.obs["array_row"] = positions["array_row"].to_numpy(dtype=int)
adata.obs["array_col"] = positions["array_col"].to_numpy(dtype=int)

# Full-resolution image coordinates
adata.obsm["spatial"] = positions[
    ["pxl_row_in_fullres", "pxl_col_in_fullres"]
].to_numpy(dtype=float)

# Load Space Ranger images/scalefactors
with open(os.path.join(spatial_path, "scalefactors_json.json")) as f:
    scalefactors = json.load(f)

hires = plt.imread(
    os.path.join(spatial_path, "tissue_hires_image.png")
)

lowres = plt.imread(
    os.path.join(spatial_path, "tissue_lowres_image.png")
)

adata.uns["spatial"] = {
    "SLV14": {
        "images": {
            "hires": hires,
            "lowres": lowres
        },
        "scalefactors": scalefactors,
        "metadata": {}
    }
}

# Minimal 2 µm filtering
sc.pp.filter_genes(adata, min_cells=3)
sc.pp.filter_cells(adata, min_counts=1)

print(adata)
print(adata.obs[["array_row", "array_col"]].head())
print("Missing rows:", adata.obs["array_row"].isna().sum())
print("Missing cols:", adata.obs["array_col"].isna().sum())
print("Spatial shape:", adata.obsm["spatial"].shape)

AnnData object with n_obs × n_vars = 1816752 × 17406
    obs: 'in_tissue', 'array_row', 'array_col', 'n_counts'
    var: 'gene_ids', 'feature_types', 'n_cells'
    uns: 'spatial'
    obsm: 'spatial'
    layers: None (.X)
                     array_row  array_col
s_002um_02449_02420       2449       2420
s_002um_01597_00975       1597        975
s_002um_02587_02503       2587       2503
s_002um_02498_02808       2498       2808
s_002um_01751_01251       1751       1251
Missing rows: 0
Missing cols: 0
Spatial shape: (1816752, 2)


#### Run Bin2Cell

In [ ]:
b2c.check_array_coordinates(adata)

b2c.destripe(adata, adjust_counts = False)


from scipy.sparse import diags

scaling = (
    adata.obs["n_counts_adjusted"].to_numpy()
    / adata.obs["n_counts"].to_numpy()
)

adata.X = diags(scaling).dot(adata.X)

adjusted_sums = np.asarray(adata.X.sum(axis=1)).ravel()

print(
    np.allclose(
        adjusted_sums,
        adata.obs["n_counts_adjusted"].to_numpy()
    )
)

target = adata.obs["n_counts_adjusted"].to_numpy()
observed = np.asarray(adata.X.sum(axis=1)).ravel()

diff = observed - target

print("Max absolute difference:", np.max(np.abs(diff)))
print("Mean absolute difference:", np.mean(np.abs(diff)))
print("Max relative difference:", np.max(np.abs(diff) / target))

print(pd.DataFrame({
    "target": target[:10],
    "observed": observed[:10],
    "difference": diff[:10]
}))




True
Max absolute difference: 1.1368683772161603e-13
Mean absolute difference: 1.0859327710146441e-15
Max relative difference: 9.862176295375152e-16
      target   observed    difference
0   4.620252   4.620252  0.000000e+00
1   9.853327   9.853327  0.000000e+00
2  19.791301  19.791301  3.552714e-15
3   1.955116   1.955116  0.000000e+00
4  13.428236  13.428236  3.552714e-15
5  13.111824  13.111824  1.776357e-15
6   6.184494   6.184494  0.000000e+00
7   9.034482   9.034482  0.000000e+00
8   1.747781   1.747781  0.000000e+00
9  14.663879  14.663879  0.000000e+00


: 

In [ ]:
mpp = 0.3

b2c.grid_image(
    adata,
    "n_counts_adjusted",
    mpp=0.3,
    sigma=5,
    save_path="../stardist/gex.tiff"
)

b2c.stardist(
    image_path="../stardist/gex.tiff",
    labels_npz_path="../stardist/gex_labels.npz",
    stardist_model="2D_versatile_fluo",
    prob_thresh=0.05,
    block_size=2048
)

b2c.insert_labels(
    adata,
    labels_npz_path="../stardist/gex_labels.npz",
    basis="array",
    mpp=0.3,
    labels_key="labels_gex"
)

cdata = b2c.bin_to_cell(
    adata,
    labels_key="labels_gex"
)